<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## Generate SMILES molecules using a Variational AutoEncoder

**Goal** 
- Gain hands-on experience with `torch` in handling simple VAE and the generation process.
- Understand the probabilistic modeling theory behind it
    
This notebook applies VAE to generate new SMILES string from the ZINC250k dataset of (canonized) strings. The network is trained on the data, and used autoregressively to actually generate new SMILES representations that are then assessed.
    
**The network will learn to 'speak' SMILES & a lower dimensional representation in a latent space**

</div>    

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training dataset**

* ~100,000 molecules represented by SMILES strings, from the 'ZINC250k` (curated) dataset of canonized strings (a curated drug-like subset of ZINC15, widely used in ML generative chemistry)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
## Why an VAE?


Implement a sequence-to-sequence VAE for SMILES: encoder, to latent vector, to decoder
- train with reconstruction loss + KL divergence

- sample new molecules from latent space

- explore latent interprolation between molecules

- evaliuate validity, uniqueness and novelty

- Show what a latent space is and how it enables controlled molecular generation: **latent space** good at exploration, interpolation and optimization tasks. THis is dimensionality reduction: nearby points correspond to similar molecules (think of Diffusion map)
- Demonstrate confidence with a **probabilistic generative model** vs purely autoregressive model (RNN)

</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Some notes on probability theory
Assume $x$ is a random variable described by density $p_{x}$, $z$ is a random variable descrived by density $p_z$. We can use marginalization to cast $p_x$ in terms of the the conditional probability: $$p_x(x) = \int p_z(z) p(x|z) dz$$
We might not know $p_x$, but can still draw samples from it by ancestrally sampling the integral: $$z \sim p_z(z), x \sim p(x|z) \Longleftrightarrow x \sim p_{x}(x)$$

In the context of generative models, $x$ represent a data sample and $z$ represents a **latent variable**, and $p_z$ is the prior. 

Approximating the conditional probability $p(x|z)$ can be intractable. Note that any other (normalized) function $q_{\Phi}(x|z)$ would still define a proper probability density upon marginalization. So, could we optimize the parameters of the model in such a way that (somehow) $q(x|z) \rightarrow p(x|z)$, ensuring that $\int p_z(z) q_\Phi(x|z) dz \rightarrow p_x$

A neural network $NN_{\theta}(\cdot)$ can be used to approximate (deterministically) a given number of parameters of the underlying distribution $p(x|z)$ (e.g., for a Gaussian, we would be approximating the mean and standard deviation); once those parameters are optimized, we can sample from the actual target distribution.

</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Variational Inference, approximate posterior and the ELBO
Assume $x$ indicates an observed data point, and that depends on a low-dim **latent variable** $z$. If $p_{\theta}(x, z)$ is the joint probability, then the probability of generating the observed data point is obtained by marginalizing over $z$, $$p_{\theta}(x) = \int p(z) p_{\theta}(x|z) dz,$$ $p_{\theta}(x|z)$ being the likelihood and $p(z)$ being the prior. Ideally, one would like to sample from this probability, but the integral is intractable.

So, introduce a more tractable distribution $q_{\Phi}(z|x)$. One can show that the following variational bound holds
$$\log p_{\theta}(x) \geq ELBO = \langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} - KL(q_{\Phi}(z|x) || p(z)),$$ which we call the **Evidence Lower Bound (ELBO)**, and that $$\log p_{\theta}(x) = ELBO + KL\left( q_{\Phi}(z|x) || p_{\theta}(x|z)\right), $$ where the **posterior** has been evidenced. Please note that when $q_{\Phi}(z|x) = p_{\theta}(x|z)$, then **ELBO** approximates exactly the data probability. Based on the variational bound, the goal is then to maximize **ELBO** $$(\hat{\theta}, \hat{\Phi}) = \arg \max_{(\theta, \Phi)} ELBO$$

Strategically, one usually selects a convenient prior $p(z)$ and also a convenient auxiliary function $q_{\Phi}$ (see below). 

**Takeaway:** The ELBO provides a tractable surrogate objective that balances two trends:  
- reconstruction accuracy via $\langle \log p_\theta(x|z)\rangle_{q_\phi} $ 
- latent regularization via the KL divergence.

</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Neural Network Parameterization

The core idea of the VAE is to represent the distributions in the ELBO as neural networks and optimize their parameters so that the ELBO is maximized, giving a tractable approximation to $\log p_\theta(x)$. After training, we can sample from the model by first drawing $z \sim p(z)$ and then generating a sequence from $p_\theta(x|z)$ (see later, `Generation`).

The network has two main components:

- **Encoder** $q_\phi(z|x)$:  
     * Deterministically models the parameters of the posterior distribution $p(z|x)$ (which we choose to be Gaussian)
     * For SMILES, this means mapping a full string into the parameters $\mu(x), \sigma(x)$ of a Gaussian distribution in latent space. (The encoder goes first naturally, because we begin from observed data $x$)

- **Decoder** $p_\theta(x|z)$:  
    * Deterministically models the likelihood $p(x|z)$ of the sequence conditioned on a latent variable $z$.  
    * For SMILES, the decoder takes as input the latent code $z$ **and** the prefix tokens $(x_1, \dots, x_{T-1})$, and predicts the next tokens $(x_2, \dots, x_T)$ = returns the $logits)$ of a categorical distribution.  

---
### Interpretation of 'NN represents a probability distribution'
A network represents a probability distribution, in the sense that the network outputs parameters of a distribution density that we can sample from (here: Gaussian, for the encoder, and categorical = discrete random variable that can take one of $K$ possible values, for the decoder). In the SMILES decoder, every softmax output is exactly that — a categorical distribution over the next possible token. In principle, other choices are possible, depending on our design choice.

The encoder network by itself only outputs two vectors of real numbers. 
We *interpret* these outputs as the mean and variance of a Gaussian distribution 
over latent codes. This interpretation is not automatic—it comes from the 
probabilistic model we chose to build (VAE). Without that assumption, the 
encoder would just be an autoencoder mapping inputs to arbitrary numbers (a 'fancy compressor'), 
without generative semantics.

Sampling from the prior and then decoding is what introduces stochasticity (and generative power) to the model. If both the encoder and decoder were fully deterministic, $$ z = f_\phi(x), x = g_\theta(z),$$ then we would have a regular autoencoder for compression.

___
### A key insight
The decoder is structurally equivalent to a **standard RNN language model**, with one important extension: it is conditioned on a global latent fingerprint $z$. Since SMILES are sequential data, RNNs (or their modern replacements like Transformers) are a natural choice for this autoregressive modeling.
</div>

In [ ]:
import numpy as np, pandas as pd, os, math, random, time
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, QED

import matplotlib
%matplotlib inline

import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

from utils import encode, decode, define_vocabulary, decode_one_smile_from_z

In [ ]:
pad_length = 120     # this will have to be tuned according to the SMILES that we have
print('Max length of input SMILES = ' + str(pad_length))

# NN TRAINING
train_fraction = 0.8
val_fraction = 0.1

batch_size = 128
print('Batch size = ' + str(batch_size))

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Curate SMILES dataset: load and canonize
    
We canonicalized all SMILES using RDKit and verified that no duplicates were removed, confirming the dataset was already distributed in canonical form. This step ensures that train/val/test splits are made on unique molecules, eliminating potential data leakage. Here, the dataset is alredy canonicalized, and all molecules are truly different from one another
    
</div>

In [ ]:
datafile = '250k_rndm_zinc_drugs_clean_3.csv'

input_data = os.path.join('data', datafile)

# Load CSV
df = pd.read_csv(input_data)
print(df.columns)    # printing the head
      
# Grab 'smiles' column
smiles_list = df["smiles"].dropna().tolist()

# Quick validity check with RDKit
valid_smiles = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        valid_smiles.append(Chem.MolToSmiles(mol, canonical=True))     # make sure there are no duplicates

print(f"We now select the first 100,000 valid SMILES molecules, to be used in the notebook")
random.shuffle(valid_smiles)
subset = valid_smiles[:100000]

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Tokenize, define inputs to VAE, prepare `DataLoader` objects for training, testing and validation
    
</div>

In [ ]:
def make_input_target(encoded):
    
    """
    Generate the encoder input, decoder input and target sequences, 
    from an encoded SMILES string
    """
    
    enc_in = encoded[:]     # input to ENCODER sees full sequence, to map to latent space
    dec_in = encoded[:-1]   #  (part of ) input to DECODER start with (<START, x_1, ... , x_{T}) (together with the latent z)
    tgt    = encoded[1:]       # target for DECODER training (x_1, ... , x_{T}, <END>)
    return enc_in, dec_in, tgt

# Build padded arrays for each split
def pad_batch(triplets, pad_id):
    
    """
    Convert a batch of (encoder, decoder, target) token sequences into
    padded NumPy arrays with consistent lengths.

    Each element of `triplets` is expected to be a tuple of three lists:
        - encoder input sequence (list of int tokens)
        - decoder input sequence (list of int tokens, e.g. prefix with <START>)
        - target sequence (list of int tokens, e.g. shifted outputs with <END>)

    This function:
      1. Finds the maximum sequence length in the encoder side (T)
         and in the decoder/target side (Td).
      2. Pads all sequences in the batch to those lengths using the given `pad_id`.
      3. Returns rectangular NumPy arrays suitable for conversion into tensors.

    Args:
        triplets (list of tuples): Batch of (encoder, decoder, target) sequences.
        pad_id (int): Token ID used for padding shorter sequences.

    Returns:
        enc_X (np.ndarray): Padded encoder input array of shape (B, T),
                            where B = batch size and T = max encoder length.
        dec_X (np.ndarray): Padded decoder input array of shape (B, Td),
                            where Td = max decoder length.
        Y     (np.ndarray): Padded target array of shape (B, Td).

    Notes:
        - Padding ensures all rows have the same length, which is required
          for batching in PyTorch/TensorFlow.
        - Standard naming convention: X = input, Y = output/labels.
    """
    
    encs = [np.array(t[0], dtype=np.int64) for t in triplets]
    decs = [np.array(t[1], dtype=np.int64) for t in triplets]
    tars = [np.array(t[2], dtype=np.int64) for t in triplets]
    T = max(map(len, encs))     #?
    Td = max(map(len, decs))    #?
    B = len(triplets)           # number of samples in batch

    def pad_to(arrs, Twant):
        X = np.full((B, Twant), pad_id, np.int64)
        for i,a in enumerate(arrs):
            X[i,:len(a)] = a
        return X

    enc_X = pad_to(encs, T)      # (B,T)
    dec_X = pad_to(decs, Td)     # (B,Td)
    Y     = pad_to(tars, Td)     # (B,Td)
    
    return enc_X, dec_X, Y       # standard notation: X is input, Y is output

In [ ]:
# --- User-defined SMILES vocabulary/alphabet: ideally, one could build this from analyzing the strings ---
# Build dictionaries
vocab, stoi, itos = define_vocabulary() 

V        = len(stoi)    # dictionary size
print('Dictionary length for tokenization = ' + str(V))

In [ ]:
# integer corresponding to the padding entry
pad_id = stoi["<PAD>"]

print('Prepare input nd output arrays')
#test_smiles = ["CCO", "CC(=O)O", "HCl"]

triplets = [make_input_target(encode(s, stoi)) for s in subset]
enc_X, dec_X, Y = pad_batch(triplets, pad_id)

print(enc_X.shape)
print(dec_X.shape)
print(Y.shape)

In [ ]:
dataset = TensorDataset(torch.from_numpy(enc_X), torch.from_numpy(dec_X), torch.from_numpy(Y))
N = len(dataset)

n_train = int(train_fraction * N)
n_val   = int(val_fraction * N)
n_test  = N - n_train - n_val

print(f"train={n_train}, val={n_val}, test={n_test}")
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],generator=torch.Generator().manual_seed(42))  # reproducible split

# data loader
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### VAE loss for SMILES Generation
As discussed already, the loss function here is $\mathcal{L}(x) = \langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} - KL(q_{\Phi}(z|x) || p(z))$ and we look to maximize that with respect to network parameters $(\Phi, \theta)$. Remember that both $x$ and $z$ are actual high-dimensional vectors, and should be denoted by $\mathbf{x}$ and $\mathbf{z}$.

- **KL term:** The KL term can be computed exactly from the assumption that $p(z)= \mathcal{N}(0,\mathbb{I})$ and $q_{\Phi}(z|x) = \mathcal{N}(z; \mu_{\Phi}(x), \sigma_{\Phi}^2(x))$ (Gaussians), i.e.
$$KL(q_{\Phi}(z|x) || p(z)) = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 -\mu_j^2 -\sigma^2_j \right)$$
where $J$ is the dimension of the latent space


- **Reconstruction term:** What about the reconstruction term? $\langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} = \int dz q_{\Phi}(z|x) \log p_{\theta}(x|z)$, which is totally impractical. We can draw $z$ samples from $q_{\Phi}$ distribution, and then use the Monte Carlo approximation:
$$\langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} \approx \frac{1}{N}\sum_{i=1}^N \log p_{\theta}(x|z_i), \quad z_i \sim q_{\Phi} $$
So that
$$\mathcal{L}(x) = \frac{1}{N}\sum_{i=1}^N \log p_{\theta}(x|z_i \sim q_{\Phi}) -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 -\mu_j^2 -\sigma^2_j \right)$$

Now, for practical reasons, one chooses $N=1$ for each data-point, and we woudl still get an unbiased estimator of the integral

**Autoregressive factorization for SMILES**: Specifically for the SMILES generation problem now, $x_{<t} = (x_1, \ldots, x_{T-1})$ is an (encoded) prefix string, so that $$ p_{\theta}(x|z) = \prod_t p_{\theta}(x_t | x_{<t}, z),$$ so the probability of the next token is conditioned on the full prefix and the value of the latent variable $z$. 
- During training, $(y_1, \ldots, y_{T-1}) = (x_2, \ldots, x_{T})$ provides the targets.
- The decoder outputs logits at each step, which are softmaxed into probabilities
- The cross-entropy compares predicted distributions to the one-hot ground truth tokens, i.e. $\sum_t y_t \log p_{\theta}(x_t | x_{<t}, z)$

### Final perspective
Thus, the reconstruction term reduces to a **standard autoregressive cross-entropy loss**, while the KL term enforces that latent codes \(z\) live in a smooth Gaussian latent space. This is what makes the model **generative** rather than just memorizing training sequences.

Upon optimization, the decoder outputs parameters of a categorical distribution that maximally overlaps with the actual underlying conditional probability $p(x|z)$: so we can say that the optimized network decoder in a sense represents the actual likelihood.

Note that, differently from a vanilla autoencoder, we are not optimizing a deterministic error, but optimize a stochastic objective (Monte-Carlo estimate of ELBO)

</div>

In [ ]:
def kl_gaussian(mu, logvar):
    """ 
    Compute the KL divergence between the Gaussian encoder and the Gaussian latent prior
    KL(N(mu, diag(exp(logvar))) || N(0,I)) per sample = -0.5 * sum(1 + logσ² - μ² - σ²)
    """
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)  # (B,)

def vae_loss(logits, y_target, pad_id, mu, logvar, beta=1.0):

    """
    VAE loss = reconstruction cross-entropy (masked over PAD) + beta * KL(q||p).

    Args:
        logits   (FloatTensor): (B, T, V) decoder logits.
        y_target (LongTensor):  (B, T)   target token ids (PAD included).
        pad_id   (int):         padding token id to ignore in CE.
        mu       (FloatTensor): (B, z_dim) encoder mean.
        logvar   (FloatTensor): (B, z_dim) encoder log-variance.
        beta     (float):       KL weight (for β-VAE / KL warm-up).

    Returns:
        loss        (Tensor): scalar total loss for backprop.
        recon_mean  (float):  mean reconstruction CE (per token, per sample).
        kl_mean     (float):  mean KL divergence.

    Note: for complex datasets, at the beginning of training (when the rec error is poor), the model often
    learns to make q \approx p(z) and gets stuck in this local minimum. To fix this, in practice it make sense to modify
    the loss function by scaling the KL divergence term by a factor \beta, where \beta is slowly annealed from 0 to 1
    """
    
    B,T,V = logits.shape
    
    # Reconstruction term: token-level cross-entropy (ignore PAD)
    ce = F.cross_entropy(logits.view(-1, V), y_target.reshape(-1),
                         ignore_index=pad_id, reduction='none'      # there will be some reduction to a scalar at some point
                        ).view(B, T)
    
    # mask out PAD positions fully
    mask = (y_target != pad_id).float()
    recon = (ce * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)  # per-sample CE, so that long smiles do not dominate
    
    # KL term per sample
    kl = kl_gaussian(mu, logvar)  # (B,)
    # Total (mean over batch): this is the EMLB averaged over the batch
    loss = recon.mean() + beta * kl.mean()
    
    return loss, recon.mean().item(), kl.mean().item()

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Architecture of our Variational Autoencoder (VAE)


The model learns to **compress** each SMILES string into a compact latent representation and then **reconstruct** it back, forming a continuous, generative embedding space for molecules.

---

**1. Embedding Layer**
- Converts each token ID (an integer symbol in the SMILES vocabulary) into a continuous vector.
- This provides the GRU with dense, meaningful representations of molecular symbols rather than sparse one-hot encodings.

**2. Encoder GRU**
- Reads the sequence of embeddings token-by-token.
- Summarizes the entire molecule into a single fixed-length hidden representation — like a “summary fingerprint” of the molecule.
- Only the **final hidden state** of the GRU is used as this compressed representation.

**3. Latent Distribution Heads**
- Two linear layers map the encoder’s hidden state into:
  - The **mean** ($\mu$)
  - The **log-variance** ($\log \sigma^2$)
- Together, these define a Gaussian distribution $q_\phi(z|x)$ — the encoder’s *belief* about which latent vector $z$ could have generated the molecule.

**4. Reparameterization Trick**
- This enables **differentiable sampling**, allowing gradients to flow through the stochastic layer during training.

**5. Mapping $z$ to the Decoder Hidden State**
- The latent vector $z$ is passed through a learned linear transformation (`z_to_h0`) followed by a `tanh` activation.
- This produces the **initial hidden state** of the decoder GRU, initializing it in a way that depends on the molecule’s latent embedding.

**6. Decoder GRU**
- Generates the SMILES sequence *autoregressively*: it predicts the next token given all previously generated ones.
- During training, **teacher forcing** is used — the model is fed the ground-truth previous token instead of its own prediction.
- This makes training faster and more stable.

**7. Projection Layer**
- Each decoder hidden state is mapped through a linear layer (`nn.Linear`) to produce **logits** — unnormalized scores over all tokens in the vocabulary.
- A softmax converts logits to probabilities, which enter the **cross-entropy loss** against the true tokens.

---

### **Training Objective – The ELBO**

The model maximizes the **Evidence Lower Bound (ELBO)**:
$$
\mathcal{L}(x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \text{KL}(q_\phi(z|x) \Vert p(z))
$$

- **Reconstruction Loss**: Encourages accurate sequence reconstruction (measured via cross-entropy).
- **KL Divergence**: Regularizes the latent space so $q_\phi(z|x)$ stays close to the prior $p(z) = \mathcal{N}(0,I)$, enforcing a smooth, continuous latent manifold where nearby points correspond to chemically similar molecules.

---

### **PyTorch GRU Reminder**
`nn.GRU` returns **two outputs**:
- **Output (`A`)** → hidden states at *each time step*: `(B, T, H)` → used for projection to logits.
- **Hidden (`B`)** → final hidden state(s) per layer: `(L, B, H)` → used as summary embedding (in the encoder) or for initializing the next GRU (in the decoder). This is the shortcut to the final hidden state(s) already contained in the other output

---

*Intuitively*:  
The **encoder** acts like a molecular “compressor” that turns SMILES strings into coordinates in a latent space.  
The **decoder** acts like a “molecule generator” that reconstructs valid SMILES from those coordinates — and, at inference time, can generate *new* molecules by decoding from unseen points in latent space.

</div>

In [ ]:
class SmilesVAE(nn.Module):

    """
    Variational Autoencoder (VAE) for molecular generation using SMILES strings.

    This model learns a probabilistic latent-space representation of molecules 
    from tokenized SMILES sequences.
    
    B = number of samples in batch, T = length of (padded) tokenization, V = vocabulary size

    Args:
        vocab_size (int): Number of tokens in the SMILES vocabulary.
        emb_dim (int): Dimension of token embeddings.
        enc_h (int): Hidden size of the encoder GRU.
        z_dim (int): Dimensionality of the latent space.
        dec_h (int): Hidden size of the decoder GRU.
        pad_id (int): Token ID reserved for padding (ignored in loss).
    
    Inputs:
        enc_in (LongTensor): Tensor of shape (B, T) with tokenized SMILES input.
        dec_in (LongTensor): Tensor of shape (B, T) with shifted input tokens (for so called teacher forcing).
    
    Outputs:
        logits (FloatTensor): decoder logits, predicted unnormalized scores of shape (B, T, V) 
                              over the vocabulary.
        mu (FloatTensor): encoder mean ->Mean of the latent distribution, shape (B, z_dim).
        logvar (FloatTensor): encoder log-variance -> Log-variance of the latent distribution, shape (B, z_dim).

    Example:
        >>> model = SmilesVAE(vocab_size=40, emb_dim=128, enc_h=256, dec_h=256, z_dim=64, pad_id=0)
        >>> logits, mu, logvar = model(enc_in, dec_in)
        >>> loss = vae_loss_fn(logits, target, mu, logvar, pad_id=0)
    """
    
    def __init__(self, vocab_size, pad_id=0, emb_dim=128, enc_h=256, z_dim = 64, dec_h = 256, num_layers=1, dropout=0.0):
        super().__init__()
        
        self.pad_id = pad_id
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.enc_h = enc_h
        self.dec_h = dec_h
        self.z_dim = z_dim
        self.num_layers = num_layers

        # Embedding
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)     #embedding layer, pad_id ignored during training
                                                                               # turn each token ID (int) into a continuous vector of size `emb_dim`
        # ---- Encoder  ----
        self.enc_rnn = nn.GRU(       # this is the actual RNN that will process the sequences of tokens at each time step
            input_size = emb_dim, 
            hidden_size = enc_h, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # reparametrization
        self.to_mu     = nn.Linear(enc_h, z_dim)    # just one mapping from hidden size to latent size
        self.to_logvar = nn.Linear(enc_h, z_dim)

        # --- Decoder routines ---- 
        self.z_to_h0 = nn.Linear(z_dim, dec_h * num_layers)    # need to go from latent variable z to hidden state
        
        self.dec_rnn = nn.GRU(       # this is the actual RNN that will process the sequences of tokens at each time step
            input_size = emb_dim, 
            hidden_size = dec_h, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        # from hidden state to logits
        self.proj = nn.Linear(dec_h, vocab_size, bias=True)
        
        # Xavier init for small heads 
        for m in [self.to_mu, self.to_logvar, self.z_to_h0, self.proj]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
        
    # ---- encode ----
    def encode(self, enc_in):
        """ Start out with the data, encode that, and compute the parameters of the approx Gaussian posterior """
        x_emb = self.embed(enc_in)
        _, h_N = self.enc_rnn(x_emb)     # extract the final hidden state [hiddens stae for every token, last hidden state for each sequence]
        h_last = h_N[-1]                 # the final hidden state is a summary of the whole molecule

        mu, logvar = self.to_mu(h_last), self.to_logvar(h_last)       # take hidden state and project to two vectors
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """
        z ~ N(mu, diag(exp(logvar))): latent fingerprint of the molecle, using a trick to make gradients flow
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std       # this makes sampling differentiable so gradients flow / 

    def init_dec_hidden(self, z):
        """
        Initialize hidden state from latent variabile z
        Map latnt vector z (B, z_dim) to the initial hidden state h0 of the decoder (shape num_layers, B, dec_h)
        """
        B = z.size(0)
        # Map latent vector - flattened hidden representaion
        h0_flat = torch.tanh(self.z_to_h0(z))          # (B, dec_h*num_layers), bounded in (-1,1)
        h0 = h0_flat.view(B, self.num_layers, self.dec_h).transpose(0,1).contiguous()
        return h0
        
    # --- decode full prefix with fixed h0-----
    def decode(self, dec_in, h0):   # takes in the string (x) & the hidden state (z)
        """

        Take in y_in and the hidden state h0 from the latent state, and decode back into the logits.
        Again, leaving out the latent injecting, this is just another RNN, like that we used in the RNN_SMILES notebook
        
        y_in: (B, T) tokens (teacher-forced inputs, start with <START>)
        h0:   (num_layers, B, dec_h) initial hidden
        returns logits: (B, T, V)
        """
        y_emb = self.embed(dec_in)     
        y_out, _ = self.dec_rnn(y_emb, h0)      # take the full story here, (B, T, V) logits, not just the last step, conditioned on the 
                                            # last hidden state
        logits = self.proj(y_out)
        return logits
        
    def forward(self, enc_in, dec_in):    # encoder input and decoder input

        mu, logvar = self.encode(enc_in)
        z = self.reparameterize(mu, logvar)     # from `x_in` to its `z` latent mapping 

        h0 = self.init_dec_hidden(z)      # we need to know the hidden state associated with `z`, so to 
                                                                       # decode
        logits = self.decode(dec_in, h0)     # takes `y_in` AND `z` (via the hidden state)
        return logits, mu, logvar     # one normalized score per vocab token, ready for cross-entropy and KL loss function

In [ ]:
# define model parameters
model = SmilesVAE(V, pad_id, emb_dim=128, enc_h=256, dec_h=256, z_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)


def run_epoch_vae(loader, train=True, beta=1.0, clip=1.0):
    
    model.train(train)
    # initialize temporary variables for saving losses
    total_loss, total_recon, total_kl, n = 0.0, 0.0, 0.0, 0
    
    for enc_X, dec_X, Y in loader:
        enc_X, dec_X, Y = enc_X.to(device), dec_X.to(device), Y.to(device)
        if train: 
            optimizer.zero_grad()     # set all the grafients to zero
            
        # pass inputs to the instantiated model and run it
        logits, mu, logvar = model(enc_X, dec_X)               # (B,T,V), (B,z), (B,z)
        loss, rce, kl = vae_loss(logits, Y, pad_id, mu, logvar, beta=beta)
        
        if train:
            loss.backward()      # backpropagate and take a step of ADAM 
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            
        bs = enc_X.size(0)
        total_loss  += loss.item() * bs
        total_recon += rce * bs
        total_kl    += kl * bs
        n += bs

    return total_loss/n, total_recon/n, total_kl/n

train_losses, val_losses = [], []

epochs = 20
best_val = float("inf")

for ep in range(1, epochs+1):
    beta = min(1.0, ep / 10)  # annealing schedule for scaling \beta
    tr = run_epoch_vae(train_loader, train=True, beta=beta)
    va = run_epoch_vae(val_loader,   train=False, beta=beta)
    
    train_losses.append(tr[0])
    val_losses.append(va[0])
    
    print(f"epoch {ep:02d} | beta {beta:.2f} | "
          f"train loss {tr[0]:.3f} (recon {tr[1]:.3f}, KL {tr[2]:.3f}) | "
          f"val loss {va[0]:.3f} (recon {va[1]:.3f}, KL {va[2]:.3f})")

    if va[0] < best_val - 1e-4:
        best_val = va[0]
        torch.save(model.state_dict(), os.path.join('results', "smiles_vae.pt"))


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Testing
    
</div>

In [ ]:
model.eval()
test_loss, test_recon, test_kl = run_epoch_vae(test_loader, train=False, beta=1.0)

print(f"\nTest set performance:")
print(f"Total ELBO loss: {test_loss:.4f}")
print(f"Reconstruction: {test_recon:.4f}")
print(f"KL divergence:  {test_kl:.4f}")

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Use trained decoder to generate samples

Sampling new points requires sampling from $p_\theta (x) = \int p(z) p_{\theta}(x|z) dz$. We will sample from the latent space $z \sim \mathcal{N}(0,1)$ and decode.

One does not have to compute the marginal $p_\theta(x)$ explicitly. Instead, sample "ancestrally" along the factorization of the joint probability $p_\theta(x,z) = p(z) p_\theta(x|z)$, which is the generative story. The marginal is defined as the integral of the joint probability. When we ignore $z$ after the sampling, we are effectively sampling from the marginal (marginalization comes for free by ignoring latent variables).

So, sampling from the marginal $p_\theta(x)$ involves two steps:

- Sample a latent:  $x \sim p(z) = \mathcal{N}(0,1)$
- Sample a sequence autoregressively from the decoder $x \sim p_\theta(x|z) = \prod_t p_\theta (x_t | x_{<t}, z)$ (this is the same function used in the `RNN_SMILES` notebook)

</div>

That way you are sampling from the generative model’s prior and letting the decoder map those points in latent space into realistic molecular strings

In [ ]:
# generate smiles

smiles_samples = []

for _ in range(n_gen):
    z = torch.randn(1, z_dim, device=device)

    # decode to a single SMILES
    s = decode_one_smile_from_z(z, max_len=120, temperature=0.9, top_k=30)

    smiles_samples.append(s)

<div style="background-color: lightyellow; padding: 10px; border-radius: 5px;">
    
## Outlook: 

* Explore the latent space using PCA, UMAP or t-SNE
* As in the `RNN_SMILES.ipynb` notebook, we would then use the same machinery to test **novelty, validity and uniqueness** of the generated molecules, and maybe visualize them using RKDit

</div>